# InfraVision: train SR → colorizer → YOLO on Colab

Trains the three models of the pipeline in order and saves each one to Google Drive as soon as it finishes,
then copies them into the repo's `checkpoints/` layout and zips them for the laptop.

| stage | data | output |
|---|---|---|
| 1. Super-resolution (EDSR) | POP thermal frames | `checkpoints/sr/edsr_best.pth` + `metrics.json` |
| 2. Colorizer (thermal → RGB) | `paired_disaster` (LLVIP pairs) | `checkpoints/colorizer/colorizer_best.pth` + `metrics.json` |
| 3. Person detector (YOLO11s) | POP | `checkpoints/yolo/best.pt` + `thresholds.json` |

**Before running** (one time, on the laptop):
1. Zip the POP folder: right-click `POP` → *Send to → Compressed folder* → `POP.zip`.
2. Pack the paired set (≈4 GB instead of ≈25 GB): `python paired_data/pack_for_colab.py`
   → `data/paired_disaster_colab.zip`.
3. Upload both zips to **Google Drive → `My Drive/InfraVision/`**.
   Optional: upload the FLIR YOLO model as `My Drive/InfraVision/thermal_best.pt` to start detection from it.

Then **Runtime → Change runtime type → T4 GPU** and **Runtime → Run all**.

A stage whose checkpoint is already on Drive is skipped, so if Colab disconnects, reconnect and *Run all* again.
Set `SMOKE = True` to check everything in a few minutes first.

In [ ]:
# ---- settings ----
SMOKE = False                 # True: 2 tiny epochs per model, to check the notebook end to end
BRANCH = "main"               # repo branch to train from
DRIVE_DIR = "/content/drive/MyDrive/InfraVision"
FORCE = []                    # stages to retrain even if Drive has them, e.g. ["sr"]

SR_EPOCHS, COLOR_EPOCHS, YOLO_EPOCHS = 30, 40, 60
YOLO_IMGSZ = 1024             # POP people are 20-40 px tall; 640 loses many of them

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, pathlib
os.makedirs(DRIVE_DIR, exist_ok=True)
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
%cd /content
!rm -rf InfraVision && git clone -q -b {BRANCH} https://github.com/elzasabu/InfraVision.git
%cd /content/InfraVision
# Colab already has torch/torchvision with CUDA; add the rest.
!pip -q install "ultralytics==8.4.191" "scikit-image" "pyyaml" "opencv-python-headless"
!git log --oneline -1

In [ ]:
# ---- data: unzip from Drive to the paths the trainers expect ----
import zipfile, glob
def unzip(src, dst="/content/InfraVision"):
    print("unzipping", src)
    with zipfile.ZipFile(src) as z:
        z.extractall(dst)

pop_zip = f"{DRIVE_DIR}/POP.zip"
assert os.path.exists(pop_zip), f"upload POP.zip to {DRIVE_DIR}"
if not os.path.isdir("POP/train"):
    unzip(pop_zip)
    # POP.zip made by "Send to compressed folder" contains a top-level POP/ folder.
    assert os.path.isdir("POP/train"), "POP.zip should contain POP/train, POP/val, POP/test"

pd_zip = f"{DRIVE_DIR}/paired_disaster_colab.zip"
if os.path.exists(pd_zip) and not os.path.exists("data/paired_disaster/manifest.csv"):
    unzip(pd_zip, "/content/InfraVision/data")
for s in ("train", "val", "test"):
    print(s, "POP frames:", len(glob.glob(f"POP/{s}/JPEGImages/*")),
          "| paired thermal:", len(glob.glob(f"data/paired_disaster/{s}/thermal/*")))

In [ ]:
def run(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode:
        raise SystemExit(f"failed: {cmd}")

def stage(name, ckpt, cmd):
    drive_out = f"{DRIVE_DIR}/checkpoints{'_smoke' if SMOKE else ''}/{name}"
    if os.path.exists(f"{drive_out}/{ckpt}") and name not in FORCE:
        print(f"{name}: already on Drive ({drive_out}/{ckpt}), skipping")
        return drive_out
    run(cmd + f" --out {drive_out}")
    return drive_out

smoke = " --smoke" if SMOKE else ""

## 1. Super-resolution (EDSR on POP, best by val PSNR)

In [ ]:
sr_dir = stage("sr", "edsr_best.pth",
               f"python -m infravision.train.sr --epochs {SR_EPOCHS} --amp --workers 2{smoke}")
!cat {sr_dir}/metrics.json | head -20

## 2. Colorizer (L1 + VGG perceptual, best by val PSNR)

Trained on the clean LLVIP pairs; POP has no RGB, so POP is judged by eye.

In [ ]:
if os.path.exists("data/paired_disaster/manifest.csv"):
    col_dir = stage("colorizer", "colorizer_best.pth",
                    f"python -m infravision.train.colorizer --epochs {COLOR_EPOCHS} --amp --workers 2{smoke}")
    !cat {col_dir}/metrics.json | head -20
    from IPython.display import Image, display
    samples = sorted(glob.glob(f"{col_dir}/val_samples_epoch*.png"))
    if samples: display(Image(samples[-1]))
else:
    print("No paired_disaster data: upload paired_disaster_colab.zip to train the colorizer")

## 3. Person detector (YOLO11s on POP), then tune conf/NMS on val and evaluate

In [ ]:
start = f"{DRIVE_DIR}/thermal_best.pt" if os.path.exists(f"{DRIVE_DIR}/thermal_best.pt") else "yolo11s.pt"
yolo_dir = stage("yolo", "best.pt",
                 f"python -m infravision.train.yolo train --start {start} --epochs {YOLO_EPOCHS} "
                 f"--imgsz {YOLO_IMGSZ} --batch 8 --workers 2{smoke}")
w = f"{yolo_dir}/best.pt"
n = " --max_images 40" if SMOKE else ""
if not os.path.exists(f"{yolo_dir}/thresholds.json") or "yolo" in FORCE:
    run(f"python -m infravision.train.yolo tune --weights {w} --split val --out {yolo_dir}{n}")
run(f"python -m infravision.train.yolo eval --weights {w} --split val --json {yolo_dir}/eval_val.json{n}")
run(f"python -m infravision.train.yolo eval --weights {w} --split test --json {yolo_dir}/eval_test.json{n}")
run(f"python -m infravision.train.yolo small --weights {w} --split val --sizes 1024 1280 --tiles 0 2 "
    f"--json {yolo_dir}/small_objects.json --max_images {40 if SMOKE else 300}")

## 4. Copy into the repo layout, zip for the laptop, quick demo check

In [ ]:
import shutil
root = f"{DRIVE_DIR}/checkpoints{'_smoke' if SMOKE else ''}"
for name in ("sr", "colorizer", "yolo"):
    if os.path.isdir(f"{root}/{name}"):
        shutil.copytree(f"{root}/{name}", f"checkpoints/{name}", dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("val_samples_*"))
!ls -la checkpoints/*
zip_path = shutil.make_archive(f"{DRIVE_DIR}/infravision_checkpoints{'_smoke' if SMOKE else ''}", "zip",
                               ".", "checkpoints")
print("Download", zip_path, "and unzip it in the repo folder on the laptop (it contains checkpoints/...).")

In [ ]:
# The same CLI the laptop uses, on 5 POP test frames.
!python -m infravision.infer --input POP/test/JPEGImages --out /content/demo_out --limit 5
from IPython.display import Image, display
for f in sorted(glob.glob("/content/demo_out/*_fused.png"))[:3]:
    display(Image(f, width=900))